In [1]:
import os
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError
import os
from dotenv import load_dotenv
import json, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv()

MONGO_URI_ORIG = os.getenv("MONGO_URI_COMUNICATION")  
MONGO_URI_DEST= os.getenv("MONGO_URI_DEST")  
MONGO_DB  = os.getenv("MONGO_DB")           # nombre DB

CHANNEL_TO_COLLECTION = {
    "Email_in": "email_inbound",
    "Email_out": "email_outbound",
    "conversation": "webchat",
    "whatsapp": "whatsapp",
}

CHECKPOINT_FILE = "migration_checkpoint.json"
FAILED_FILE = "migration_failed.jsonl"


In [2]:
def find_record(payload_obj, db):
    
    mongo_query = {}

    conversation_data = payload_obj['data']['item']
    channel = conversation_data.get('type', 'Unknown')
    id = conversation_data.get('id', 'Unknown')

    if channel == "email":
        mongo_query = {
            "Conversation ID":id
        } 
        channel = 'Email_in' if conversation_data.get('type', 'Unknown') == 'customer_initiated' else 'Email_out'
    else:
        mongo_query = {
            'msgid': id
        }

    collection = CHANNEL_TO_COLLECTION[channel] 

    count = db[collection].count_documents(mongo_query)
    
    return True if count > 0 else False

def get_db(uri):
    client = MongoClient(
        uri,
        serverSelectionTimeoutMS=5000,
        appname="qb-prod-migration",
    )
    # ping para validar conexión
    client.admin.command("ping")
    return client, client[MONGO_DB]

import os, json
from bson import ObjectId


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
    return ObjectId(data["last_id"]) if data.get("last_id") else None

def save_checkpoint(last_id, i):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump({"last_id": str(last_id), "i": i}, f)

def log_failed(doc_id, reason, extra=None):
    rec = {"_id": str(doc_id), "reason": reason}
    if extra:
        rec["extra"] = extra
    with open(FAILED_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [3]:
# import json
# import requests
# from requests.adapters import HTTPAdapter
# from urllib3.util.retry import Retry
# import time 

# # --- session con retries ---
# session = requests.Session()
# retries = Retry(
#     total=6,
#     backoff_factor=0.5,
#     status_forcelist=[429, 500, 502, 503, 504],
#     allowed_methods=["POST"],
#     raise_on_status=False,
# )
# session.mount("http://", HTTPAdapter(max_retries=retries))

# mongo_query = {
#     "$nor": [{"Channel": "email", "source": "customer_initiated"}]
# }

# client_orig, db_orig = get_db(MONGO_URI_ORIG)
# client_dest, db_dest = get_db(MONGO_URI_DEST)

# url = "http://127.0.0.1:8080/webhook/intercom"

# last_id = load_checkpoint()

# # filtro para reanudar
# query = dict(mongo_query)
# if last_id is not None:
#     query["_id"] = {"$gt": last_id}

# cursor = db_orig["email_inbound"].find(query, no_cursor_timeout=True).sort("_id", 1).limit(100)


# start = time.time()
# i = 0
# try:
#     for i, doc in enumerate(cursor, 1):
#         doc_id = doc["_id"]

#         raw = doc.get("raw")
#         if not raw:
#             log_failed(doc_id, "missing_raw")
#             save_checkpoint(doc_id, i)
#             continue

#         try:
#             payload_obj = raw if isinstance(raw, dict) else json.loads(raw)

#         except json.JSONDecodeError as e:
#             log_failed(doc_id, "json_decode_error", {"err": str(e)})
#             save_checkpoint(doc_id, i)
#             continue

#         try:
            
#             r = session.post(url, json=payload_obj, timeout=60)
#             if r.status_code >= 300:
#                 log_failed(doc_id, "http_error", {"status": r.status_code, "body": r.text[:500]})
#             # si existe, no hacemos nada
#         except Exception as e:
#             # cualquier error de red/timeout/etc.
#             log_failed(doc_id, "exception", {"err": str(e)})

#         # checkpoint cada N docs (más eficiente)
#         if i % 100 == 0:
#             save_checkpoint(doc_id, i)
#             print(f"Processed {i} - last_id={doc_id}")

#     # checkpoint final
#     if i > 0:
#         save_checkpoint(doc_id, i)

# finally:
#     cursor.close()
    
# end = time.time()
# print(end - start)

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- session factory (un Session por hilo) ---
def make_session():
    s = requests.Session()
    retries = Retry(
        total=6,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["POST"],
        raise_on_status=False,
    )
    s.mount("http://", HTTPAdapter(max_retries=retries, pool_connections=100, pool_maxsize=100))
    return s

URL = "http://127.0.0.1:8080/webhook/intercom"

def chunked(iterable, size):
    batch = []
    for x in iterable:
        batch.append(x)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

# Lock para escritura thread-safe en el archivo de fallos
_failed_lock = threading.Lock()
_checkpoint_lock = threading.Lock()

def log_failed_safe(doc_id, reason, extra=None):
    rec = {"_id": str(doc_id), "reason": reason}
    if extra:
        rec["extra"] = extra
    with _failed_lock:
        with open(FAILED_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def save_checkpoint_safe(last_id, i):
    with _checkpoint_lock:
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump({"last_id": str(last_id), "i": i}, f)

def process_batch(docs, db_dest):
    session = make_session()
    ok = skipped = failed = 0

    for doc in docs:
        doc_id = doc["_id"]
        raw = doc.get("raw")
        if not raw:
            log_failed_safe(doc_id, "missing_raw")
            skipped += 1
            continue

        try:
            payload_obj = raw if isinstance(raw, dict) else json.loads(raw)
        except json.JSONDecodeError as e:
            log_failed_safe(doc_id, "json_decode_error", {"err": str(e)})
            failed += 1
            continue

        try:
            r = session.post(URL, json=payload_obj, timeout=60)
            if r.status_code < 300:
                ok += 1
            else:
                log_failed_safe(doc_id, "http_error", {"status": r.status_code})
                failed += 1
        except requests.RequestException as e:
            log_failed_safe(doc_id, "exception", {"err": str(e)})
            failed += 1

    last_id = str(docs[-1]["_id"])
    return ok, skipped, failed, last_id

# --- consulta Mongo (en orden estable) ---
mongo_query = {
    "$nor": [{"Channel": "email", "source": "customer_initiated"}],
    "raw": {"$exists": True, "$ne": None},
}

client_orig, db_orig = get_db(MONGO_URI_ORIG)
client_dest, db_dest = get_db(MONGO_URI_ORIG)

cursor = db_orig["email_inbound"].find(mongo_query, no_cursor_timeout=True)

BATCH_SIZE = 10
MAX_WORKERS = 20

start = time.time()
tot_ok = tot_skip = tot_fail = 0

try:
    processed=0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = []
        for batch in chunked(cursor, BATCH_SIZE):
            futures.append(ex.submit(process_batch, batch, db_dest))

        for f in as_completed(futures):
            ok, skipped, failed, last_id = f.result()
            tot_ok += ok; tot_skip += skipped; tot_fail += failed
            processed += ok + skipped + failed
            if processed % 5000 == 0:  # checkpoint cada 5k docs
                save_checkpoint_safe(last_id, processed)
        print(f"batch done | ok={ok} skip={skipped} fail={failed} | last_id={last_id}")

finally:
    cursor.close()

print("TOTAL:", tot_ok, tot_skip, tot_fail, "secs:", round(time.time() - start, 2))

In [13]:
count = db_orig["email_inbound"].count_documents(query)
print(count)

NameError: name 'query' is not defined

In [ ]:
mongo_query = {
    "$nor": [
        {
            "Channel": "email",
            "source": "customer_initiated"
        }
    ]
}
cursor  = db['email_inbound'].find(mongo_query, no_cursor_timeout=True)